In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)


### **Read Data from Silver Layer**

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

In [0]:
df_gold = df_silver.select("product_code", "month", "gross_price")
display(df_gold.limit(5))

### **Write the Final Data Frame to Gold Schema**

In [0]:
df_gold.write \
    .format("delta") \
        .option("delat.enableChangeDataFeed", True) \
            .mode("overwrite") \
                .saveAsTable(f"{catalog}.{gold_schema}.sb_dim{data_source}")

### **Merging Child Company Data with Parent Company**

In [0]:
df_gold_price = spark.read.table(f"fmcg.gold.sb_dimgross_price")
df_gold_price.show(5)

In [0]:
df_gold_price = df_gold_price.withColumn("year", F.year(F.col("month"))) \
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))

In [0]:
display(df_gold_price.limit(5))

In [0]:
W = Window.partitionBy(F.col("product_code"), F.col("year")) \
    .orderBy(F.col("is_zero"), F.col("month").desc())

df_gold_latest_price = df_gold_price.withColumn("rnk", F.row_number().over(W)) \
    .filter(F.col("rnk") == 1)

display(df_gold_latest_price)

In [0]:
# Take required columns

df_gold_latest_price = df_gold_latest_price.withColumn("year", F.col("year").cast("string")) \
    .select(F.col("product_code"), F.col("gross_price").alias("price_inr"), F.col("year"))
    
df_gold_latest_price.show(5)


In [0]:
df_gold_latest_price.printSchema()

In [0]:
delta_table = DeltaTable.forName(spark, f"fmcg.gold.dim_gross_price")

delta_table.alias("target").merge(
    source = df_gold_latest_price.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set = {
        "price_inr" : "source.price_inr",
        "year" : "source.year"
    }
).whenNotMatchedInsert(
    values = {
        "product_code" : "source.product_code",
        "price_inr" : "source.price_inr",
        "year" : "source.year"
    }
).execute()